<a href="https://colab.research.google.com/github/andrew-veriga/Titans_jax/blob/main/colabs/Titans_jax_TPUv6e-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Gemma-Titans Training on TPU with Kauldron

This notebook demonstrates the complete pipeline for training and using the **Gemma-Titans** hybrid model on a Google Cloud TPU v5e using the `Kauldron` framework (DeepMind's standard training library).

### Steps included:
1. **Initialization:** Loading base Gemma weights and randomly initializing Titans NLTM modules using `SkipTitans`.
2. **Pre-training (CPT):** Training *only* the Titans memory layers using `kd.optim.partial_updates` to avoid TPU OOM, utilizing a proper dataset.
3. **Save/Load:** Splitting the PyTree to save only the fine-tuned memory weights.
4. **Dialogue Inference:** Running the model and updating the internal memory cache dynamically.

In [2]:
# 0. Environment Setup

# Clone the new kauldron repository
!git clone --depth 1 https://github.com/google-research/kauldron || true
!pip install -q ./kauldron
# Clone the gemma repository if not already present
!git clone --depth 1 https://github.com/google-deepmind/gemma.git || true
!pip install -q ./gemma

# Clone the dialog repository to fix Gemma format issues
!git clone --depth 1 https://github.com/google-deepmind/dialog || true
!pip install -q ./dialog

!pip install -q flax optax seqio





Cloning into 'kauldron'...
remote: Enumerating objects: 9674, done.
remote: Counting objects: 100% (201/201), done.
remote: Compressing objects: 100% (133/133), done.
remote: Total 9674 (delta 81), reused 69 (delta 68), pack-reused 9473 (from 3)
Receiving objects: 100% (9674/9674), 2.89 MiB | 275.00 KiB/s, done.
Resolving deltas: 100% (6951/6951), done.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 36.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 

In [4]:
!git clone --depth 1 https://github.com/andrew-veriga/Titans_jax.git


Cloning into 'Titans_jax'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 29 (delta 1), reused 13 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (29/29), 195.14 KiB | 470.00 KiB/s, done.
Resolving deltas: 100% (1/1), done.


In [5]:
!pip install importlib_resources

# Start

In [ ]:
import os
# os._exit(0)

In [ ]:
# !git pull

In [1]:
# Ensure our local modules are in the Python path
import sys
import os
sys.path.append(os.getcwd())
from kauldron import kd
from kauldron.ktyping import Array

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [2]:
import jax
import jax.numpy as jnp
import optax
from kauldron import kd
import numpy as np
import os
import orbax.checkpoint as ocp
import dataclasses

# Gemma imports
from gemma import gm
from gemma.gm.nn import _config

# Our custom Titans integration
import importlib

%cd Titans_jax
import gemma_titans
importlib.reload(gemma_titans)
from gemma_titans import Gemma3_1B_Titans, Gemma_Titans_Config
from titans_ckpts import SkipTitans
import titans_tree_utils

print(f"JAX Backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")
""" старые настройки
# Prevent JAX from allocating 100% of TPU memory instantly
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# Limit XLA to 85% of TPU HBM to leave room for overhead
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".85"
# Reduce fragmentation and compilation memory spike
os.environ["XLA_FLAGS"] = "--xla_gpu_enable_highest_priority_async_collectives=true --xla_tpu_enable_data_parallel_all_reduce_opt=true --xla_tpu_memory_bound_loop_fusion_limit=1"
os.environ["JAX_COMPILATION_CACHE_DIR"] = "/tmp/jax_cache"
"""
# Разрешаем JAX забрать память сразу для максимальной скорости
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "true"

# Увеличиваем долю памяти (оставляем чуть-чуть на системные нужды)
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".95"

# Оптимизируем флаги для производительности, а не для экономии
os.environ["XLA_FLAGS"] = (
    "--xla_tpu_enable_data_parallel_all_reduce_opt=true "
    "--xla_tpu_joint_all_gather_opt=true "
    "--xla_tpu_enable_latency_hiding_scheduler=true " # Скрывает задержки памяти за вычислениями
    "--xla_tpu_all_reduce_combine_threshold_bytes=134217728" # Оптимально для больших батчей
)

os.environ["JAX_COMPILATION_CACHE_DIR"] = "/tmp/jax_cache"


/content/Titans_jax
JAX Backend: tpu
Devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]


## 1. Model initialization

We load previously trained titans weights and merge them with original Gemma weights

In [8]:
!cp "/content/drive/Shareddrives/shared veriga/saved_titans_delta/saved_titans_delta.zip" saved_titans_delta.zip

In [9]:
!rm saved_titans_delta.zip

In [3]:
import shutil
import os

def load_titans_weights(load_dir: str):
    load_path = os.path.abspath(load_dir)
    checkpointer = ocp.StandardCheckpointer()

    # Restore directly without a template (returns dictionary of arrays)
    titans_params = checkpointer.restore(load_path)
    return titans_params

merged_params = None
titans_delta_path = "./saved_titans_delta"
titans_zip = "saved_titans_delta.zip" #saved_titans_delta_short_context.zip
workdir_checkpoints = "./titans_workdir_squad/checkpoints"

if os.path.exists(workdir_checkpoints) and len(os.listdir(workdir_checkpoints)) > 0:
    print(f"📁 Найдена директория {workdir_checkpoints}. Пропускаем ручное слияние весов.")
    print("Kauldron автоматически загрузит последнее состояние при старте обучения.")
else:
    if os.path.exists(titans_zip):
        print(f"Unpacking {titans_zip}...")
        shutil.unpack_archive(titans_zip, titans_delta_path)

    if os.path.exists(titans_delta_path):
        # Load original Gemma 3 1B IT weights
        print("Loading original Gemma weights...")
        original_params = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)

        # Merge loaded Titans weights into original Gemma params
        print("Merging Titans weights...")
        import titans_tree_utils
        loaded_titans_params = load_titans_weights(titans_delta_path)
        merged_params = titans_tree_utils.merge_titans_params(original_params, loaded_titans_params)
        print("✅ Success: Pre-trained Titans weights loaded and merged.")
    else:
        print("⚠️ Note: './saved_titans_delta' not found. Titans layers will be randomly initialized.")

📁 Найдена директория ./titans_workdir_squad/checkpoints. Пропускаем ручное слияние весов.
Kauldron автоматически загрузит последнее состояние при старте обучения.


## 2. Pre-training (CPT)

We use `Kauldron` to orchestrate the training.
- **Dataset:** Instead of a python generator, we use `kd.data.py.Elements`.
- **Model Config:** We use the standard `gm.nn.Gemma3_1B.config`.
- **SkipTitans:** Handles loading Gemma weights while keeping Titans randomized.
- **partial_updates:** Ensures XLA only builds backprop graphs for memory parameters to prevent OOM.

### hyper-parameters

In [4]:
# Create a proper Kauldron dataset pipeline using TFDS
batch_size = 8
max_length = 2048 #
total_steps = 20000



In [5]:
# Define the model configuration
import dataclasses

gemma_config = dataclasses.replace(
    Gemma3_1B_Titans.config,
    titans_layer_indices=(11, 17, 23)  # Уменьшаем размер окна для слоев с памятью
)

# Initialize model
model = Gemma3_1B_Titans(
    config=gemma_config, # По умолчанию is_training_mode=True
    dtype=jnp.bfloat16,
    return_last_only=False,
    tokens="batch.tokens",
)

def format_squad(x):
    # SQuAD structure: context, question, answers (dict with 'text' list)
    answers = x.get('answers', {}).get('text', [])
    answer = answers[0] if len(answers) > 0 else "Unknown"

    # Robustly handle potential bytes from TFDS
    ctx = x["context"].decode('utf-8') if hasattr(x["context"], 'decode') else x["context"]
    q = x["question"].decode('utf-8') if hasattr(x["question"], 'decode') else x["question"]
    ans = answer.decode('utf-8') if hasattr(answer, 'decode') else answer

    # Create 'src' and 'dst' for the Seq2SeqTask
    x['src'] = "Context: " + ctx + "\n\nQuestion: " + q
    x['dst'] = ans
    return x
def format_triviaqa(x):
    # В trivia_qa/rc структура: 'entity_pages' (list) или 'search_results'
    # Берем первый попавшийся контекст из entity_pages
    ctx = ""
    # if x['entity_pages']['wiki_context']:
    #     ctx = x['entity_pages']['wiki_context'][0]
    if x['search_results']['search_context']:
        ctx = x['search_results']['search_context'][0]
    q = x["question"]
    # Ответ в TriviaQA обычно в x['answer']['value']
    ans = x['answer']['value']

    x['src'] = f"Context: {ctx}\n\nAnswer the question in one sentence: {q}"
    x['dst'] = ans
    return x

tokenizer = gm.text.Gemma3Tokenizer()


In [6]:
# import importlib
# # importlib.reload(adam_atan2)
# from adam_atan2 import adam_atan2

## dataset

In [7]:
# 1. Выбираем TyDi QA
import grain
MAX_CONTEXT_CHARS = (max_length - 50) * 4  # ~3700 символов для max_length=1024

class FilterShortContext(grain.python.FilterTransform):
    def filter(self, x):
        ctxs = x['search_results']['search_context']
        if not ctxs:
            return False
        return len(ctxs[0]) <= MAX_CONTEXT_CHARS

def get_train_dataset_tydi_qa():
    return kd.data.py.Tfds(
        name= "trivia_qa/rc", #splits: 'test'	17,210 | 'train'	138,384 | 'validation'	18,669
        split="train",
        shuffle=True,
        num_epochs=None,
        batch_size=batch_size,
        num_workers=1,
        transforms=[
            # поля context/question/answers
            FilterShortContext(),
            format_triviaqa, # format_squad,
            gm.data.Seq2SeqTask(
                in_prompt="src",
                in_response="dst",
                out_input="tokens",
                out_target="target",
                tokenizer=tokenizer,
                max_length=max_length,
                truncate=True,
            ),
            kd.data.py.Elements(keep=["tokens", "target"]),
        ],
    )




### загрузка отфильтрованного датасета с короткими контекстами

In [8]:
os.path.abspath('/content/drive/Shareddrives/shared veriga/trivia_qa_filtered')

'/content/drive/Shareddrives/shared veriga/trivia_qa_filtered'

In [9]:
import pickle

DATA_DIR = os.path.abspath('/content/drive/Shareddrives/shared veriga/trivia_qa_filtered')

def get_precomputed_dataset(batch_size=16, tokenizer=None, max_length=1024, files=None):
    """Загружает и объединяет отфильтрованные данные из нескольких файлов.

    Args:
        batch_size: Размер батча.
        tokenizer: Токенизатор Gemma.
        max_length: Максимальная длина последовательности.
        files: Список имен файлов, например ['train.array_record', 'validation.array_record']
    """
    if files is None:
        files = ['train_gemma_generated.array_record']

    paths = [os.path.join(DATA_DIR, f) for f in files]
    print(f"Загрузка данных из: {paths}")

    # 1. Определяем источник данных (DataSource)
    class PickledArrayRecordDataSource(grain.python.ArrayRecordDataSource):
        def __getitem__(self, idx):
            return pickle.loads(super().__getitem__(idx))

    data_source = PickledArrayRecordDataSource(paths)

    # 2. Создаем Kauldron Dataset
    return kd.data.py.DataSource(
        seed=42,
        data_source=data_source,
        shuffle=True,      # Перемешивание важно при объединении разных сплитов!
        num_epochs=None,   # Бесконечно для обучения
        batch_size=batch_size,
        transforms=[
            format_triviaqa, # format_squad,
            gm.data.Seq2SeqTask(
                in_prompt="src",
                in_response="dst",
                out_input="tokens",
                out_target="target",
                out_target_mask="loss_mask",
                tokenizer=tokenizer,
                max_length=max_length,
                truncate=True,
            ),
            kd.data.py.Elements(keep=["tokens", "target"]),
        ],
    )

In [10]:
print(dir(grain.python))

['ArrayRecordDataSource', 'Batch', 'BatchOperation', 'DATASET_INDEX', 'DataLoader', 'DatasetIterator', 'DatasetSelectionMap', 'EPOCH', 'FilterOperation', 'FilterTransform', 'INDEX', 'InMemoryDataSource', 'IndexSampler', 'IterDataset', 'META_FEATURES', 'MapDataset', 'MapOperation', 'MapTransform', 'MapWithIndexTransform', 'MultiprocessingOptions', 'NoSharding', 'Operation', 'PyGrainCheckpointHandler', 'PyGrainCheckpointRestore', 'PyGrainCheckpointSave', 'PyGrainDatasetIterator', 'RECORD', 'RECORD_KEY', 'RandomAccessDataSource', 'RandomMapOperation', 'RandomMapTransform', 'RangeDataSource', 'ReadOptions', 'Record', 'RecordMetadata', 'SEED', 'Sampler', 'SequentialSampler', 'ShardByJaxProcess', 'ShardOptions', 'SharedMemoryArray', 'Transformation', 'Transformations', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'config', 'experimental', 'load']


### optimizer routing

In [12]:
from adam_atan2 import adam_atan2
from optax.contrib._muon import MuonDimensionNumbers

def muon_only_dims(params):
    MUON_KEYS = {'to_queries', 'to_keys_values', 'combine_heads'}
    def _label(path, v):
        key    = str(path[-1].key) if hasattr(path[-1], 'key') else ''
        parent = str(path[-2].key) if len(path) > 1 and hasattr(path[-2], 'key') else ''
        if key == 'kernel' and parent in MUON_KEYS:
            return MuonDimensionNumbers(reduction_axis=0, output_axis=1)
        return None
    return jax.tree_util.tree_map_with_path(_label, params)

def is_muon_param(path_str):
    parts = path_str.split('/')
    return (len(parts) >= 2 and
            parts[-1] == 'kernel' and
            parts[-2] in {'to_queries', 'to_keys_values', 'combine_heads'})

def muon_mask(params):
    def _m(path, v):
        return is_muon_param('/'.join(str(p.key) for p in path))
    return jax.tree_util.tree_map_with_path(_m, params)

def is_gate_param(path_str):
    # Проверяем, относится ли параметр к гейту
    return 'memory_gate' in path_str.split('/')

def gate_mask(params):
    def _m(path, v):
        return is_gate_param('/'.join(str(p.key) for p in path))
    return jax.tree_util.tree_map_with_path(_m, params)

def adam_base_mask(params):
    def _m(path, v):
        path_str = '/'.join(str(p.key) for p in path)
        return not is_muon_param(path_str) and not is_gate_param(path_str)
    return jax.tree_util.tree_map_with_path(_m, params)



# Сниженные LR: веса уже предобучены в Phase 1
# lr_muon = optax.warmup_cosine_decay_schedule(
#     init_value=1e-4,
#     peak_value=5e-4,
#     warmup_steps=500,
#     decay_steps=total_steps - 500,
#     end_value=1e-5,
# )
lr_muon = optax.cosine_decay_schedule(
    init_value=1e-5,
    decay_steps=total_steps,
    alpha=0.05,  # конечный lr = 1e-3 * 0.05 = 5e-5
)
# lr_adam = optax.warmup_cosine_decay_schedule(
#     init_value=1e-4,
#     peak_value=5e-4,
#     warmup_steps=500,
#     decay_steps=total_steps - 500,
#     end_value=1e-5,
# )
lr_adam = optax.cosine_decay_schedule(
    init_value=1e-5,
    decay_steps=total_steps,
    alpha=0.05,
)

lr_gate = optax.cosine_decay_schedule(
    init_value=1e-3,  # В 100 раз больше, чем lr_adam (1e-5)
    decay_steps=total_steps,
    alpha=0.05,
)

## lr for experiments 4e-4, 1.5e-4
inner_chain = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.masked(
        optax.contrib.muon(
            learning_rate=lr_muon,
            muon_weight_dimension_numbers=muon_only_dims,
            # beta=0.80,
            eps=1e-8,
            mu_dtype=jnp.float32,
        ),
        mask=muon_mask,
    ),
    # 2. Adam для гейтов (с высоким LR)
    optax.masked(
        adam_atan2(
            learning_rate=lr_gate,
            b1=0.90, b2=0.90,
            eps=1e-8,
            mu_dtype=jnp.float32
        ),
        mask=gate_mask,
    ),
    # 3. Adam для остальных параметров (с обычным LR)
    optax.masked(
        adam_atan2(
            learning_rate=lr_adam,
            b1=0.90,
            b2=0.80,
            eps=1e-8,
            mu_dtype=jnp.float32
        ),
        mask=adam_base_mask,
    ),
    )

routing_optimizer = optax.MultiSteps(
    kd.optim.partial_updates(
        inner_chain,
        mask=kd.optim.select(["memory", "memory_gate"]),
    ),
    every_k_schedule=16,
)

### metrics

In [13]:
# --- INITIALIZATION LOGIC ---
# We define a simple class because Kauldron expects an object with a .transform() method
import flax
from kauldron import metrics as kd_metrics
from kauldron import kd
from kauldron import kontext

class FullParamsInit(kd.ckpts.InitTransform):
    def __init__(self, params):
        self.params = params
    def transform(self, state: kd.train.TrainState) -> kd.train.TrainState:
        return state.replace(params=self.params)

class TPUMemoryMetric(kd_metrics.Metric):
    """Метрика для логирования использования памяти TPU в ГБ."""
    @flax.struct.dataclass
    class State(kd_metrics.State):
        # Состояние может быть пустым, так как данные мы берем напрямую из JAX на хосте
        def compute(self):
            stats_dict = {}
            for i, device in enumerate(jax.devices()):
                try:
                    m_stats = device.memory_stats()
                    # Если словарь пустой, пропускаем устройство
                    if not m_stats:
                        continue
                    prefix = f"device_{i}"
                    # 1. Текущее использование памяти (если ключа нет, вернет 0)
                    if 'bytes_in_use' in m_stats:
                        used_gb = m_stats['bytes_in_use'] / 1e9
                        stats_dict[f"{prefix}/used_gb"] = np.array(used_gb, dtype=np.float32)
                    # 2. Пиковое использование
                    if 'peak_bytes_in_use' in m_stats:
                        peak_gb = m_stats['peak_bytes_in_use'] / 1e9
                        stats_dict[f"{prefix}/peak_gb"] = np.array(peak_gb, dtype=np.float32)
                    # 3. Лимит и процент (только если 'limit' действительно существует)
                    if 'limit' in m_stats and 'bytes_in_use' in m_stats:
                        limit_gb = m_stats['limit'] / 1e9
                        usage_pct = (m_stats['bytes_in_use'] / m_stats['limit']) * 100
                        stats_dict[f"{prefix}/usage_pct"] = np.array(usage_pct, dtype=np.float32)
                except (AttributeError, ValueError, RuntimeError):
                    pass
            return stats_dict
        @classmethod
        def empty(cls):
            """Создает пустое начальное состояние."""
            return cls()
        def merge(self, other):
            """Объединяет состояния из разных батчей (здесь ничего не делаем)."""
            return self

    def get_state(self, **kwargs) -> State:
        # Просто возвращаем пустое состояние.
        # Нам не нужны данные из batch или модели для этой метрики.
        return self.State().empty()

@dataclasses.dataclass(kw_only=True, frozen=True, eq=True)
class MemoryGateMetric(kd_metrics.Metric):
    """Метрика для логирования средних значений и степени открытия memory_gate."""

    # Просто указываем строку пути в контексте (Kauldron сам всё поймет)
    params: kd.kontext.Key = "params"

    @flax.struct.dataclass
    class State(kd_metrics.State):
        gate_metrics: flax.core.FrozenDict[str, jnp.ndarray] = flax.core.FrozenDict()

        def compute(self):
            return {k: np.array(v, dtype=np.float32) for k, v in self.gate_metrics.items()}

        @classmethod
        def empty(cls):
            return cls(gate_metrics=flax.core.FrozenDict())

        def merge(self, other):
            return self

    def get_state(self, params=None, **kwargs) -> State:
        if params is None:
            return self.State.empty()

        stats_dict = {}

        def find_gates(tree, path_prefix=""):
            if hasattr(tree, 'items'):
                for key, val in tree.items():
                    current_path = f"{path_prefix}_{key}" if path_prefix else str(key)
                    if key == "memory_gate":
                        mean_val = jnp.mean(val)
                        openness = jax.nn.sigmoid(mean_val)
                        stats_dict[f"titans_gates/{current_path}_raw_mean"] = mean_val
                        stats_dict[f"titans_gates/{current_path}_openness"] = openness
                    else:
                        find_gates(val, current_path)

        find_gates(params)  # ← реальное дерево, не строка
        return self.State(gate_metrics=flax.core.freeze(stats_dict))

        return self.State(gate_metrics=flax.core.freeze(stats_dict))

# Генерируем словарь для Kauldron

Titan_losses = {
# Kauldron понимает синтаксис [ключ] для доступа к словарям внутри PyTree
    f"distill_layer_{i}": kd.losses.Value(
        values=f"preds.layer_losses['loss_layer_{i}']"
    )
    for i in gemma_config.titans_layer_indices
}

# 2. Метрики для графиков (собирают сырой MSE)
Titan_metrics = {
    # f"raw_mse_{i}": kd.metrics.SingleDimension(
    #     tensor=f"preds.layer_losses['raw_mse_layer_{i}']", # Тут аргумент называется tensor, а не values
    #     index=None # Отключаем срез по индексу, чтобы класс взял всё число целиком
    # )
    # for i in gemma_config.titans_layer_indices
}
Titan_metrics["tpu_memory"]= TPUMemoryMetric()
Titan_metrics["memory_gate"] = MemoryGateMetric()
# Titan_metrics["learning_rate"] = kd.metrics.StateValue(key="optimizer.hyperparams.learning_rate")
import flax.struct
import jax.numpy as jnp
# 1. Описываем, как хранить значение Learning Rate
@flax.struct.dataclass
class LRState(kd.metrics.State):
    lr_value: jnp.ndarray
    @classmethod
    def empty(cls):
        return cls(lr_value=jnp.array(0.0))
    def merge(self, other):
        # При объединении батчей/шагов просто берем последнее значение
        return self #cls(lr_value=other.lr_value)
    def compute(self):
        return self.lr_value
# 2. Создаем саму метрику
@dataclasses.dataclass(kw_only=True, frozen=True, eq=True)
class LearningRateMetric(kd.metrics.Metric):
    step: kontext.Key = "step"

    def get_state(self, step, **kwargs):
        # Вычисляем LR для текущего шага
        # ВАЖНО: Замените `lr_schedule` на имя вашей переменной расписания
        # Если у вас просто фиксированное число (например, 0.001), напишите: current_lr = 0.001
        current_lr = lr_muon(step)
        return LRState(lr_value=jnp.array(current_lr))

Titan_metrics["learning_rate"] = LearningRateMetric()

In [14]:
import flax.linen as nn
import grain
# Configure Trainer

"""
train_ds = get_precomputed_dataset(
    batch_size=batch_size,
    tokenizer=tokenizer,
    max_length=max_length,
    files=['train.array_record', 'validation.array_record', 'test.array_record']
),
"""

trainer = kd.train.Trainer(
    seed=42,
    workdir=os.path.abspath('./titans_workdir_squad'),
    # train_ds= get_train_dataset_tydi_qa(), ##get_train_dataset(),
    train_ds = get_precomputed_dataset(
        batch_size=batch_size,
        tokenizer=tokenizer,
        max_length=max_length,
        files=[
            # 'validation_gemma_generated.array_record',
            # 'test_gemma_generated.array_record',
            'train_gemma_generated.array_record'
            ]
    ),
    model= model,
    # If we have merged_params from Cell 1, use them directly (much faster).
    # Otherwise, use SkipTitans to load Gemma and randomly init Titans.
    init_transform = FullParamsInit(merged_params) if merged_params is not None else SkipTitans(
        wrapped=gm.ckpts.LoadCheckpoint(
            path=gm.ckpts.CheckpointPath.GEMMA3_1B_IT,
        ),
        workdir=os.path.abspath('./SkipTitans_workdir')
    ),
    num_train_steps = total_steps, #87600 // 16, #dataset length times to epoches num divide to batch_size
    train_losses=Titan_losses,
    train_metrics=Titan_metrics,
    optimizer = routing_optimizer,

    checkpointer=kd.ckpts.Checkpointer(save_interval_steps=500),

    # Sharding for TPU / TPU Pods
    # sharding=kd.sharding.ShardingStrategy(),
    # train_metrics={

    #     # Наш монитор памяти
    #     "tpu_memory": TPUMemoryMetric()
    # },
)

print("Trainer initialized. Starting compilation and training...")


Загрузка данных из: ['/content/drive/Shareddrives/shared veriga/trivia_qa_filtered/train_gemma_generated.array_record']
Trainer initialized. Starting compilation and training...


In [15]:
Titan_metrics

{'tpu_memory': <__main__.TPUMemoryMetric at 0x784de7d1dc10>,
 'memory_gate': MemoryGateMetric(params='params'),
 'learning_rate': LearningRateMetric(step='step')}

### 2.5 Monitoring with TensorBoard
Launch TensorBoard to monitor training metrics (Loss, Learning Rate) in real-time.

In [16]:
%reload_ext tensorboard


In [17]:
from tensorboard import notebook

# Показать список всех активных инстансов
notebook.list()

No known TensorBoard instances running.


In [18]:
!rm -rf /tmp/.tensorboard-info/*

In [ ]:
%tensorboard --logdir ./titans_workdir_squad/ --port=6006

## Start training

In [ ]:
# import importlib
# importlib.reload(gemma_titans)
# from gemma_titans import Gemma3_1B_Titans, Gemma_Titans_Config


In [26]:
# run the actual training loop:
state, aux = trainer.train()

Disabling pygrain multi-processing (unsupported in colab).
Starting training loop at step 0


train:   0%|          | 0/20001 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
import jax
import gc

# Clear JAX's internal live buffer cache
# This is necessary because JAX often keeps buffers alive for potential reuse
for device in jax.devices():
    device.live_arrays().clear() if hasattr(device, 'live_arrays') else None
    device.default_memory_tracker().clear() if hasattr(device, 'default_memory_tracker') else None
    device.live_buffers().clear() if hasattr(device, 'live_buffers') else None
jax.clear_caches()
# Run Python garbage collection
gc.collect()

print("TPU memory cache cleared and garbage collection finished.")

In [ ]:
trainer = trainer.replace(num_train_steps=55000)

state, aux = trainer.train()


In [ ]:
import os
# os._exit(0)

## 3. Save / Load Trained Weights
We don't want to save the entire 1B model, just our new memory projections. We utilize the `split_titans_params` utility.

In [ ]:
# !cd /content/Titans_jax/titans_workdir_squad/checkpoints && zip -r /content/Titans_jax/ckpt_24000.zip ckpt_24000

# Проверим, что файл создался и посмотрим его размер
# !ls -lh /content/Titans_jax/ckpt_24000.zip

In [ ]:
def save_titans_weights(state: kd.train.TrainState, save_dir: str):
    # 1. Extract only the Titans weights
    _, titans_params = titans_tree_utils.split_titans_params(state.params)

    import shutil
    save_path = os.path.abspath(save_dir)
    if os.path.exists(save_path):
        shutil.rmtree(save_path)

    checkpointer = ocp.StandardCheckpointer()

    # 2. Correct Orbax syntax: save(path, state)
    # Passing titans_params as the positional state saves only them.
    checkpointer.save(save_path, titans_params)
    checkpointer.wait_until_finished()
    print(f"Saved ONLY Titans weights to {save_path}")

def load_titans_weights(load_dir: str):
    load_path = os.path.abspath(load_dir)
    checkpointer = ocp.StandardCheckpointer()

    # Restore directly without a template (returns dictionary of arrays)
    titans_params = checkpointer.restore(load_path)
    return titans_params

# Usage:
save_titans_weights(state, "./saved_titans_delta")
loaded_titans_params = load_titans_weights("./saved_titans_delta")
original_params, _ = titans_tree_utils.split_titans_params(state.params)
state = state.replace(params=titans_tree_utils.merge_titans_params(original_params, loaded_titans_params))

def save_training_checkpoint(state: kd.train.TrainState, workdir: str):
    """
    Полное сохранение состояния обучения (веса + оптимизатор + шаг).
    Позволяет возобновить тренировку с того же места.
    """
    import orbax.checkpoint as ocp
    checkpoint_path = os.path.join(workdir, 'checkpoints')

    # Kauldron уже имеет встроенный чекпоинтер в Trainer,
    # но этот метод показывает, как сделать это вручную через Orbax.
    options = ocp.CheckpointManagerOptions(max_to_keep=3, create=True)
    mngr = ocp.CheckpointManager(os.path.abspath(checkpoint_path), ocp.StandardCheckpointer(), options)

    # Сохраняем весь объект state (включая opt_state и step)
    mngr.save(state.step, state)
    mngr.wait_until_finished()
    print(f"✅ Full training checkpoint saved at step {state.step} to {checkpoint_path}")

def load_training_checkpoint(state_template: kd.train.TrainState, workdir: str) -> kd.train.TrainState:
    """
    Восстановление полной сессии обучения.
    """
    import orbax.checkpoint as ocp
    checkpoint_path = os.path.join(workdir, 'checkpoints')

    if not os.path.exists(checkpoint_path):
        print("⚠️ No checkpoint found. Starting from scratch.")
        return state_template

    mngr = ocp.CheckpointManager(os.path.abspath(checkpoint_path), ocp.StandardCheckpointer())
    latest_step = mngr.latest_step()

    if latest_step is None:
        return state_template

    # Восстанавливаем состояние, используя state_template как структуру
    restored_state = mngr.restore(latest_step, items=state_template)
    print(f"🚀 Restored training session from step {latest_step}")
    return restored_state

In [ ]:
!ls ./saved_titans_delta


In [ ]:
import os
import zipfile
import shutil
from google.colab import files

checkpoint_dir = "./saved_titans_delta"

# First, let's check if the directory exists
if not os.path.exists(checkpoint_dir):
    print(f"Error: {checkpoint_dir} does not exist!")
else:
    # Try Colab download first
    try:

        # Create a zip file of the checkpoint
        zip_filename = "saved_titans_delta.zip"

        # Remove old zip if exists
        if os.path.exists(zip_filename):
            os.remove(zip_filename)

        # Create zip file
        shutil.make_archive("saved_titans_delta", 'zip', checkpoint_dir)
        !cp ./saved_titans_delta.zip "/content/drive/Shareddrives/shared veriga/saved_titans_delta/saved_titans_delta.zip"

        # Download the zip file
        files.download(zip_filename)
        print(f"✓ Downloaded {zip_filename} via Colab")

    except ImportError:
        # Not in Colab, create zip and provide manual download instructions
        print("Not running in Google Colab. Creating zip file...")

        zip_filename = "saved_titans_delta.zip"

        # Remove old zip if exists
        if os.path.exists(zip_filename):
            os.remove(zip_filename)

        # Create zip file
        shutil.make_archive("saved_titans_delta", 'zip', checkpoint_dir)

        # Get file size
        file_size = os.path.getsize(zip_filename) / (1024 * 1024)  # Size in MB

        print(f"\n✓ Created {zip_filename} ({file_size:.2f} MB)")
        print(f"📁 Full path: {os.path.abspath(zip_filename)}")
        print("\nTo download, use one of these methods:")
        print("1. Right-click the file in Jupyter file browser and select 'Download'")
        print("2. Use scp/rsync from your terminal:")
        print(f"   scp user@server:{os.path.abspath(zip_filename)} ./")

In [ ]:
# 6. Пуш в GitHub
from google.colab import userdata
try:
    GITHUB_PAT = userdata.get('GITHUB_PAT')
    repo_url_raw = get_ipython().getoutput('git config --get remote.origin.url')
    if not repo_url_raw:
            print("❌ Не удалось получить URL репозитория git.")
            # return

    repo_url = repo_url_raw[0].replace("https://", "").replace("git@github.com:", "").replace(".git", "")
    auth_url = f"https://{GITHUB_PAT}@{repo_url}.git"

    get_ipython().system('git config --global user.email "colab-emergency@google.com"')
    get_ipython().system('git config --global user.name "Colab Emergency Bot"')
    get_ipython().system(f'git add saved_titans_delta_short_context.zip')
    get_ipython().system(f'git commit -m "Compact backup: Step {total_steps} [automated]" || echo "No changes"')
    get_ipython().system(f'git push {auth_url} HEAD:main --force')

    print(f"🚀 Бэкап {zip_filename} (~100MB) успешно отправлен в GitHub.")
except Exception as e:
    print(f"❌ Ошибка GitHub: {e}")


## КОМПАКТНОЕ ЭКСТРЕННОЕ СОХРАНЕНИЕ В GITHUB

In [ ]:
# --- КОМПАКТНОЕ ЭКСТРЕННОЕ СОХРАНЕНИЕ В GITHUB ---
# Сохраняет только ПОСЛЕДНИЙ чекпоинт и логи TensorBoard.
# Размер архива составит ~100 МБ вместо 4 ГБ.

import os
import shutil
import orbax.checkpoint as ocp
from google.colab import userdata

# def emergency_save_and_push(
workdir='./titans_workdir_squad'
    # ):
checkpoint_root = os.path.join(workdir, 'checkpoints')
if not os.path.exists(checkpoint_root):
    print(f"❌ Директория {checkpoint_root} не найдена.")
    # return

# 1. Находим последний шаг через Orbax (ИСПРАВЛЕНО: убран legacy-аргумент ocp.StandardCheckpointer())
mngr = ocp.CheckpointManager(os.path.abspath(checkpoint_root))
latest_step = mngr.latest_step()
"""
if latest_step is None:
    print("❌ Актуальные чекпоинты не найдены.")
    return

print(f"🔍 Найден последний шаг: {latest_step}")

# 2. Создаем временную папку для чистого экспорта
export_dir = "./titans_export_temp"
if os.path.exists(export_dir):
    shutil.rmtree(export_dir)
os.makedirs(export_dir)

# 3. Копируем только ПОСЛЕДНИЙ шаг и метаданные Orbax
os.makedirs(os.path.join(export_dir, "checkpoints"))
step_src = os.path.join(checkpoint_root, str(latest_step))
step_dst = os.path.join(export_dir, "checkpoints", str(latest_step))

if os.path.exists(step_src):
    shutil.copytree(step_src, step_dst)
else:
    print(f"❌ Папка шага {step_src} не найдена.")
    # return

# Копируем служебные файлы Orbax, необходимые для загрузки
for meta in ['_checkpoint_manager_state.json', 'metadata']:
    meta_src = os.path.join(checkpoint_root, meta)
    if os.path.exists(meta_src):
        dest = os.path.join(export_dir, "checkpoints", meta)
        if os.path.isdir(meta_src):
            shutil.copytree(meta_src, dest)
        else:
            shutil.copy(meta_src, dest)

# 4. Копируем логи TensorBoard (tfevents)
for item in os.listdir(workdir):
    if item.startswith('events.out.tfevents'):
        shutil.copy(os.path.join(workdir, item), export_dir)

# 5. Архивируем только экспортную папку
zip_name = "titans_compact_backup.zip"
if os.path.exists(zip_name):
    os.remove(zip_name)

print(f"📦 Создание компактного архива...")
shutil.make_archive("titans_compact_backup", 'zip', export_dir)
shutil.rmtree(export_dir)

# 6. Пуш в GitHub
try:
    GITHUB_PAT = userdata.get('GITHUB_PAT')
    repo_url_raw = get_ipython().getoutput('git config --get remote.origin.url')
    if not repo_url_raw:
            print("❌ Не удалось получить URL репозитория git.")
            # return

    repo_url = repo_url_raw[0].replace("https://", "").replace("git@github.com:", "").replace(".git", "")
    auth_url = f"https://{GITHUB_PAT}@{repo_url}.git"

    get_ipython().system('git config --global user.email "colab-emergency@google.com"')
    get_ipython().system('git config --global user.name "Colab Emergency Bot"')
    get_ipython().system(f'git add {zip_name}')
    get_ipython().system(f'git commit -m "Compact backup: Step {latest_step} [automated]" || echo "No changes"')
    get_ipython().system(f'git push {auth_url} HEAD:main --force')

    print(f"🚀 Бэкап {zip_name} (~100MB) успешно отправлен в GitHub.")
except Exception as e:
    print(f"❌ Ошибка GitHub: {e}")

# emergency_save_and_push()
"""

In [ ]:
options = ocp.CheckpointManagerOptions()
with ocp.CheckpointManager(os.path.abspath(workdir + '//'),options) as mngr:
    latest_step = mngr.latest_step()
    print(latest_step)

In [ ]:
osp.test_utils.